In [23]:
import yfinance as yf
import pandas as pd
import sqlite3 as db

ticker = yf.Ticker("GOOGL")

# get historical market data
df = ticker.history(start = "2024-01-01", end = "2025-12-31" , interval = "1h")

new_df = ticker.get_news()

# drop the column Dividends and Stock Splits
df = df.drop(columns = ['Dividends', 'Stock Splits'])



# add the day name to the data frame as a new column
df['Day'] = df.index.day_name()
df['hour'] = df.index.hour


# print the row that have open price greater than 1500
#print(df[df.Open > 1600])

#print(df.head())

# format the new_df , it's in list of dictionaries
new_df = pd.DataFrame(new_df)

# the columns of the new_df is Index(['id', 'content'], dtype='object')
# print the content of the first 10 news
print(new_df)
# print(new_df['content'][0]) print each title in a new line


#print(new_df.columns)


                                     id  \
0  fa1100b5-3ca6-49ad-88a0-da553a98a5e1   
1  e75ca192-cbdb-4d69-ac6f-0c4216496e8a   
2  07dd468b-14a2-3620-bb7d-a27531d9fa67   
3  b8a213f3-0c21-3737-b698-4f8aa8ebecbb   
4  9928da9c-133e-3f3f-b094-fb423f28f17f   
5  1b7e42e8-ff6e-3ce9-92af-aab9641f00b5   
6  9e7bcbef-694d-392c-876f-dabc201418a8   
7  b7356628-cf0c-33f5-9106-7ee4dc2aacde   
8  7debda62-756b-3e5e-b904-d41a5bd0e61c   
9  13ae0e9c-7fd1-31d0-858d-4b3358c4350d   

                                             content  
0  {'id': 'fa1100b5-3ca6-49ad-88a0-da553a98a5e1',...  
1  {'id': 'e75ca192-cbdb-4d69-ac6f-0c4216496e8a',...  
2  {'id': '07dd468b-14a2-3620-bb7d-a27531d9fa67',...  
3  {'id': 'b8a213f3-0c21-3737-b698-4f8aa8ebecbb',...  
4  {'id': '9928da9c-133e-3f3f-b094-fb423f28f17f',...  
5  {'id': '1b7e42e8-ff6e-3ce9-92af-aab9641f00b5',...  
6  {'id': '9e7bcbef-694d-392c-876f-dabc201418a8',...  
7  {'id': 'b7356628-cf0c-33f5-9106-7ee4dc2aacde',...  
8  {'id': '7debda62-756b-3e5e-b

In [27]:
# apply a filters to the data
filter = (df.index.day_name() == 'Monday') & (df.Open > 1600)
df.loc[filter]

,Open,High,Low,Close,Volume,Day
Date,,,,,,
2020-02-24 00:00:00-05:00,1657.000000,1686.599976,1650.000000,1672.400024,435,Monday
2020-03-09 00:00:00-04:00,1696.099976,1701.599976,1657.699951,1674.500000,161,Monday
2020-03-30 00:00:00-04:00,1641.199951,1652.800049,1607.199951,1622.000000,16389,Monday
2020-04-06 00:00:00-04:00,1629.099976,1696.699951,1625.900024,1677.000000,1063,Monday
2020-04-13 00:00:00-04:00,1722.000000,1756.800049,1710.699951,1744.800049,696,Monday
...,...,...,...,...,...,...
2024-12-02 00:00:00-05:00,2649.000000,2649.800049,2621.699951,2634.899902,695,Monday
2024-12-09 00:00:00-05:00,2632.100098,2677.100098,2630.800049,2664.899902,935,Monday
2024-12-16 00:00:00-05:00,2658.300049,2663.300049,2651.000000,2651.399902,877,Monday


In [28]:
# this will print the mean of the close price between 2020-01-03 and 2020-01-10
print(df.loc['2020-01-03':'2020-01-10']["Close"].mean()) 


1558.9666544596355


In [36]:
# resample the data to get the mean of the close price for each month
#print(df.resample('M').mean(numeric_only =True))

# more complex aggregation using max and min and mean for each month per column
print(df.resample('M').agg({'Open':'max', 'High':'min', 'Low':'mean', 'Close':'mean'}))

                                  Open         High          Low        Close
Date                                                                         
2020-01-31 00:00:00-05:00  1584.300049  1528.699951  1552.633347  1559.933338
2020-02-29 00:00:00-05:00  1657.000000  1560.699951  1586.073685  1595.615780
2020-03-31 00:00:00-04:00  1696.099976  1484.000000  1573.831815  1594.650008
2020-04-30 00:00:00-04:00  1756.800049  1600.000000  1677.714292  1694.980940
2020-05-31 00:00:00-04:00  1755.699951  1701.500000  1708.584991  1718.690015
2020-06-30 00:00:00-04:00  1779.800049  1699.400024  1724.763644  1735.922735
2020-07-31 00:00:00-04:00  1963.400024  1784.000000  1836.204540  1847.540905
2020-08-31 00:00:00-04:00  2045.500000  1928.500000  1945.933332  1966.238084
2020-09-30 00:00:00-04:00  1967.300049  1857.699951  1912.347621  1920.276187
2020-10-31 00:00:00-04:00  1924.300049  1876.199951  1889.527266  1900.245450
2020-11-30 00:00:00-05:00  1955.599976  1787.199951  1856.425000

C:\Users\yoonus\AppData\Local\Temp\ipykernel_18120\315560953.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  print(df.resample('M').agg({'Open':'max', 'High':'min', 'Low':'mean', 'Close':'mean'}))


In [7]:
# create a connection to the database
conn = db.connect('Data/data.db')

# create table for the data
df.to_sql('Google', conn, if_exists = 'replace')

# close the connection to the database
conn.commit()
conn.close()

In [11]:
# fetch the data from the database from google table
conn = db.connect('Data/data.db')
df = pd.read_sql('SELECT * FROM Google where High > 140', conn)
print(df.head())

                    Datetime        Open        High         Low       Close  \
0  2024-01-09 09:30:00-05:00  138.500000  140.039795  138.149994  139.711304   
1  2024-01-09 10:30:00-05:00  139.720001  140.625000  139.330002  140.410004   
2  2024-01-09 11:30:00-05:00  140.404999  141.360001  140.309998  141.289993   
3  2024-01-09 12:30:00-05:00  141.279999  141.485001  140.979996  141.154999   
4  2024-01-09 13:30:00-05:00  141.149994  141.195007  140.699997  140.820007   

    Volume      Day  hour  
0  5582594  Tuesday     9  
1  3066557  Tuesday    10  
2  2640449  Tuesday    11  
3  2225419  Tuesday    12  
4  2579403  Tuesday    13  
